# SmolVLM2-500M → Robo2VLM-1 fine-tune (SmolVLA backbone), full train split, multi-session

**How to run across sessions (Kaggle 12 h limit)**
1. Session 1: *Save Version → Save & Run All*. The script stops itself at `MAX_HOURS` (11 h), writes `last/` and exits cleanly.
2. Session 2: *Add Input → Your Work → Notebook Output* → pick session 1's output. Run the notebook again **with `FRESH_START = False`** - it finds `last/` under `/kaggle/input/...` and continues at the exact step and data position.
3. Keep `MAX_TRAIN_SAMPLES`, `PER_DEVICE_BS`, `GRAD_ACCUM` identical in every session (they define the LR schedule). The script refuses to resume otherwise.
4. Optional insurance: add a Kaggle secret `HF_TOKEN` and set env `HUB_REPO=user/name` - `last/`/`best/` are uploaded on exit and used for resume if no local copy exists.

In [ ]:
!pip install -q -U num2words bitsandbytes datasets transformers tensorboard matplotlib psutil


In [ ]:
%%writefile /kaggle/working/train_v3.py
"""
Full fine-tune HuggingFaceTB/SmolVLM2-500M-Video-Instruct on keplerccc/Robo2VLM-1  (Kaggle 2x T4, DDP)
Goal: a VLM backbone that is a drop-in for SmolVLA (LeRobot) -> robot-aware spatial / interaction features.

What is different from v2
  * SmolVLA-compatible: LLM truncated to the first KEEP_LLM_LAYERS (16, what SmolVLA uses), 512x512 letterboxed input,
    one tile / 64 visual tokens per image, vision tower trained here (SmolVLA freezes it later).
  * Whole train split (678,034 rows = 1 epoch) across several Kaggle sessions. Resume is exact: model + optimizer +
    scheduler + the position inside the data stream (per rank). A restart neither replays the epoch nor skips ahead.
  * Safe multi-session: finds the previous session's last/ in /kaggle/input (or on the HF Hub), never wipes it,
    carries best/ + logs over, skips the final test when stopped by the clock, time measured from process start.
  * Host RAM: no DataLoader workers (a background thread prefetches instead), small streaming batches,
    glibc/arrow allocator tweaks, RSS logged every LOG_EVERY steps.
  * Loss only on the answer letter (v2 also supervised the "\nAssistant:" tokens, which diluted the loss ~4x).
    Running train accuracy is logged next to validation accuracy (overfitting / shortcut check).
  * Fixed validation split (same rows in every session) so best/ selection is comparable across sessions.

Launch: torchrun --standalone --nproc_per_node=2 train_v3.py
"""
import os, io, re, ast, gc, json, csv, time, glob, copy, queue, ctypes, shutil, random, threading, datetime, logging, contextlib
T_START = time.time()                                    # the 12 h Kaggle clock starts before this; MAX_HOURS leaves margin
from itertools import islice
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")           # curbs glibc RSS growth
os.environ.setdefault("ARROW_DEFAULT_MEMORY_POOL", "system")   # pyarrow's pool otherwise keeps freed parquet buffers
os.environ.setdefault("HF_HOME", "/tmp/hf")

import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from PIL import Image
from datasets import load_dataset, Image as HFImage
from datasets.distributed import split_dataset_by_node
from transformers import AutoProcessor, AutoModelForImageTextToText, get_cosine_schedule_with_warmup

# ----------------------------------------------------------------------------- config
MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
DATASET = "keplerccc/Robo2VLM-1"
OUT_DIR = Path("/kaggle/working/robo2vlm_smolvlm2_run")

TRAIN_ROWS = 678_034             # size of the train split
MAX_TRAIN_SAMPLES = 678_034      # 1 epoch. KEEP THIS IDENTICAL IN EVERY SESSION (it defines the LR schedule)
PER_DEVICE_BS = 4
GRAD_ACCUM = 8                   # effective batch = 4 * 2 GPUs * 8 = 64  -> 10,594 optimizer steps / epoch
LR = 2e-5
VISION_LR_MULT = 0.25            # vision tower is frozen inside SmolVLA later; keep its drift small
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 50
MAX_GRAD_NORM = 1.0
FREEZE_VISION = False            # False = true full fine-tune
OPTIM = "adamw8bit"              # "adamw8bit" or "adamw"
USE_AMP = True                   # fp16 autocast + GradScaler (T4 has no bf16)
GRAD_CKPT = True                 # try False if gpu_peak_gb stays well under ~11 GB (faster)

# --- SmolVLA compatibility
KEEP_LLM_LAYERS = 16             # SmolVLA (num_vlm_layers=16) reads only the first 16 of the 32 LLM layers. 0/None = keep all
IMAGE_SPLITTING = False          # one 512x512 tile -> 64 visual tokens, as in SmolVLA
IMG_SIZE = 512
RESIZE_MODE = "pad"              # "pad": aspect-preserving letterbox like SmolVLA's resize_with_pad | "squash": processor default

# --- eval / checkpointing
N_VAL = 1000                     # rows of `test` for model selection (500 gave +-2.2 % noise)
N_TEST = 0                       # rows of `test` after the N_VAL ones for the final test; 0 = all remaining
EVAL_EVERY = 500
SAVE_EVERY = 500                 # resumable last/ (fp32 + optimizer, ~3 GB)
LOG_EVERY = 10
EVAL_BS = 8
SPLIT_SEED, SPLIT_BUFFER = 0, 2000   # fixed forever: defines which test rows are VAL vs TEST in every session

# --- host RAM / streaming
STREAM_BATCH = 8                 # parquet record-batch rows
SHUFFLE_BUFFER = 256             # ~ tens of MB of compressed bytes; 16 would give near-sequential (correlated) batches
PREFETCH = 2                     # batches prepared by a background thread (no worker processes)

MAX_HOURS = 11.0                 # measured from process start; leaves ~1 h for last save / upload / session teardown
SEED = 42
RESUME = True
HUB_REPO = os.environ.get("HUB_REPO")     # optional private HF repo ("user/name") used as checkpoint insurance
PREV_RUN_GLOBS = ["/kaggle/input/*/robo2vlm_smolvlm2_run/last", "/kaggle/input/*/*/robo2vlm_smolvlm2_run/last",
                  "/kaggle/input/*/last", "/kaggle/working/hub_prev/last"]

# ----------------------------------------------------------------------------- dist / logging
rank = int(os.environ.get("RANK", 0)); world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
torch.cuda.set_device(local_rank); dev = torch.device("cuda", local_rank)
dist.init_process_group("nccl", timeout=datetime.timedelta(minutes=60))
is_main = rank == 0

OUT_DIR.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger("train"); logger.setLevel(logging.INFO if is_main else logging.WARNING)
fmt = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
if is_main:
    fh = logging.FileHandler(OUT_DIR / "train.log"); fh.setFormatter(fmt); logger.addHandler(fh)
log = logger.info


def all_mean(x):
    t = torch.tensor([x], device=dev, dtype=torch.float32); dist.all_reduce(t)
    return (t / world).item()


def trim_host_memory():
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass


random.seed(SEED + rank); torch.manual_seed(SEED)
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"])

# ----------------------------------------------------------------------------- processor
def make_processor(padding_side):
    p = AutoProcessor.from_pretrained(MODEL_ID, do_image_splitting=IMAGE_SPLITTING)
    p.image_processor.do_image_splitting = IMAGE_SPLITTING
    p.tokenizer.padding_side = padding_side
    return p

processor = make_processor("right")        # training
eval_processor = make_processor("left")    # generation
tok = processor.tokenizer
EOU_ID = tok.convert_tokens_to_ids("<end_of_utterance>")
IMG_ID = tok.convert_tokens_to_ids("<image>")

# ----------------------------------------------------------------------------- data
def parse_choices(c):
    if isinstance(c, (list, tuple)):
        out = [str(x) for x in c]
    else:
        s = str(c).strip(); out = None
        for fn in (json.loads, ast.literal_eval):
            try:
                v = fn(s)
                if isinstance(v, (list, tuple)): out = [str(x) for x in v]; break
                if isinstance(v, dict): out = [str(v[k]) for k in sorted(v)]; break
            except Exception:
                pass
        if out is None:
            out = [p.strip() for p in re.split(r"\n|;;", s) if p.strip()]
    return [re.sub(r"^\(?[A-Za-z][\.\):]\s+", "", x).strip() for x in out]


def build_prompt(question, choices):
    opts = "\n".join(f"{chr(65 + i)}. {c}" for i, c in enumerate(choices))
    return f"{str(question).strip()}\n\nOptions:\n{opts}\n\nAnswer with the letter of the correct option."


# <<PREP_IMG
def prep_img(img):
    """RGB + (optionally) SmolVLA-style resize_with_pad: fit inside IMG_SIZE^2 keeping aspect, black padding left/top."""
    img = img.convert("RGB")
    if RESIZE_MODE != "pad":
        return img
    w, h = img.size
    r = max(w, h) / IMG_SIZE
    nw, nh = max(1, int(w / r)), max(1, int(h / r))
    img = img.resize((nw, nh), Image.BILINEAR)
    canvas = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (0, 0, 0))
    canvas.paste(img, (IMG_SIZE - nw, IMG_SIZE - nh))
    return canvas
# PREP_IMG>>


def load_img(v):
    if isinstance(v, Image.Image): img = v
    elif isinstance(v, dict) and v.get("bytes") is not None: img = Image.open(io.BytesIO(v["bytes"]))
    else: img = Image.open(v["path"])
    return prep_img(img)


_stream_bs_logged = False
def _load(split):
    global _stream_bs_logged
    try:
        ds = load_dataset(DATASET, split=split, streaming=True, batch_size=STREAM_BATCH)
        if not _stream_bs_logged: log(f"streaming with batch_size={STREAM_BATCH}")
    except Exception as e:
        if not _stream_bs_logged: log(f"batch_size not supported ({type(e).__name__}); using default")
        ds = load_dataset(DATASET, split=split, streaming=True)
    _stream_bs_logged = True
    return ds


def make_stream(split, seed, buffer, by_node=True, shuffle=True):
    ds = _load(split).cast_column("image", HFImage(decode=False))   # raw bytes; decode in collate
    if shuffle: ds = ds.shuffle(seed=seed, buffer_size=buffer)
    if by_node and world > 1: ds = split_dataset_by_node(ds, rank=rank, world_size=world)
    return ds


def detect_answer_base():
    ds = _load("test").remove_columns(["image"])
    has0 = hit_len = False
    for ex in islice(ds, 400):
        n = len(parse_choices(ex["choices"])); a = int(ex["correct_answer"])
        has0 |= a == 0; hit_len |= a == n
    return 1 if (hit_len and not has0) else 0

ANS_BASE = detect_answer_base()


def split_rows(start, stop):
    """Deterministic, identical on every rank/run/session: rows [start, stop) of the fixed-seed shuffled test split."""
    for i, ex in enumerate(make_stream("test", SPLIT_SEED, SPLIT_BUFFER, by_node=False)):
        if i < start: continue
        if stop and i >= stop: break
        yield ex


def compress_row(ex):
    buf = io.BytesIO(); load_img(ex["image"]).save(buf, "JPEG", quality=95)
    return {"question": ex["question"], "choices": ex["choices"], "correct_answer": ex["correct_answer"],
            "image": {"bytes": buf.getvalue()}}


def collate_train(examples):
    texts, images = [], []
    for ex in examples:
        ch = parse_choices(ex["choices"]); idx = int(ex["correct_answer"]) - ANS_BASE
        if not (0 <= idx < len(ch) <= 26): continue
        msgs = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": build_prompt(ex["question"], ch)}]},
            {"role": "assistant", "content": [{"type": "text", "text": chr(65 + idx)}]},
        ]
        texts.append(processor.apply_chat_template(msgs, add_generation_prompt=False).strip())
        images.append([load_img(ex["image"])])
    if not texts: return None
    b = processor(text=texts, images=images, return_tensors="pt", padding=True)
    ids = b["input_ids"]; labels = torch.full_like(ids, -100)
    for i in range(ids.size(0)):
        pos = (ids[i] == EOU_ID).nonzero().flatten()
        if len(pos) >= 2:   # supervise ONLY the answer letter and the closing <end_of_utterance>
            e = pos[-1].item()
            labels[i, e - 1:e + 1] = ids[i, e - 1:e + 1]
        else:
            labels[i] = ids[i]
    labels[labels == IMG_ID] = -100
    labels[b["attention_mask"] == 0] = -100
    b["labels"] = labels
    return b


def collate_eval(samples):
    texts, images, gold = [], [], []
    for ex in samples:
        ch = parse_choices(ex["choices"]); idx = int(ex["correct_answer"]) - ANS_BASE
        if not (0 <= idx < len(ch) <= 26): continue
        msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": build_prompt(ex["question"], ch)}]}]
        texts.append(eval_processor.apply_chat_template(msgs, add_generation_prompt=True))
        images.append([load_img(ex["image"])]); gold.append(chr(65 + idx))
    if not texts: return None, []
    return eval_processor(text=texts, images=images, return_tensors="pt", padding=True), gold


# <<FEEDER
class Feeder:
    """
    Background-thread batch producer with an exactly resumable position.
    Every queued batch carries the datasets-library stream state (shard / row-group position) *after* that batch, so
    `self.state` always describes what the training loop has really consumed, not what the thread has prefetched.
    Resuming drops at most the shuffle-buffer content (<= SHUFFLE_BUFFER rows per rank) - rows are never replayed.
    """
    def __init__(self, state, start_epoch, make_stream_fn, collate_fn, seed, buffer, bs, prefetch, log_fn=print):
        self._mk, self._collate, self._seed, self._buf, self._bs, self._log = make_stream_fn, collate_fn, seed, buffer, bs, log_fn
        self.state = state or {"epoch": start_epoch, "ds": None, "n_rows": 0}
        self._init = copy.deepcopy(self.state)
        self._q = queue.Queue(prefetch); self._stop = threading.Event()
        self._t = threading.Thread(target=self._run, daemon=True); self._t.start()

    def _put(self, item):
        while not self._stop.is_set():
            try: self._q.put(item, timeout=1); return True
            except queue.Full: pass
        return False

    def _open(self, epoch, st):
        ds = self._mk("train", self._seed + epoch, self._buf)
        n_rows = 0
        if st and st.get("ds") is not None:
            try:
                ds.load_state_dict(st["ds"]); n_rows = st.get("n_rows", 0)
                self._log(f"data stream restored (epoch {epoch}, {n_rows} rows consumed on this rank)")
            except Exception as e:
                n_rows = st.get("n_rows", 0)
                self._log(f"load_state_dict failed ({type(e).__name__}: {e}); falling back to skip({n_rows})")
                if n_rows: ds = ds.skip(n_rows)
        return ds, n_rows

    def _run(self):
        try:
            epoch, st = self._init["epoch"], self._init
            while not self._stop.is_set():
                ds, n_rows = self._open(epoch, st); buf = []
                for ex in ds:
                    n_rows += 1; buf.append(ex)
                    if len(buf) < self._bs: continue
                    b = self._collate(buf); buf = []
                    if b is None: continue
                    if not self._put((b, {"epoch": epoch, "ds": copy.deepcopy(ds.state_dict()), "n_rows": n_rows})): return
                    del b
                self._log(f"epoch {epoch} stream exhausted on this rank -> starting epoch {epoch + 1}")
                epoch += 1; st = None
        except BaseException as e:           # surface worker errors in the training loop
            self._put(e)

    def next(self):
        item = self._q.get()
        if isinstance(item, BaseException): raise item
        batch, self.state = item
        return batch

    def close(self):
        self._stop.set()
# FEEDER>>

# ----------------------------------------------------------------------------- resume discovery (local, Kaggle input, HF Hub)
def read_meta(d):
    try: return json.loads((d / "meta.json").read_text())
    except Exception: return None


def find_resume():
    if not RESUME: return None
    if HUB_REPO and is_main and not (OUT_DIR / "last" / "meta.json").exists() and not any(glob.glob(g) for g in PREV_RUN_GLOBS):
        try:
            from huggingface_hub import snapshot_download
            snapshot_download(HUB_REPO, allow_patterns=["last/*", "best/*"], local_dir="/kaggle/working/hub_prev")
            log(f"downloaded previous checkpoint from hub repo {HUB_REPO}")
        except Exception as e:
            log(f"hub resume skipped: {type(e).__name__}: {e}")
    dist.barrier()
    cands = [OUT_DIR / "last"] + [Path(p) for g in PREV_RUN_GLOBS for p in glob.glob(g)]
    best = None
    for c in cands:
        m = read_meta(c)
        if m and (c / "trainer_state.pt").exists() and (best is None or m["step"] > best[0]["step"]):
            best = (m, c)
    return best


def adopt_previous_run(prev_last):
    """Carry best/ and the logs of the previous session into this session's OUT_DIR (never touch /kaggle/input)."""
    if not is_main or prev_last.parent.resolve() == OUT_DIR.resolve(): return
    root = prev_last.parent
    for name in ("log_history.jsonl", "log_history.csv", "config.json"):
        if (root / name).exists() and not (OUT_DIR / name).exists(): shutil.copy2(root / name, OUT_DIR / name)
    if (root / "best").exists() and not (OUT_DIR / "best").exists(): shutil.copytree(root / "best", OUT_DIR / "best")


resume = find_resume()
resume_state = resume_meta = None; resume_dir = None
if resume:
    resume_meta, resume_dir = resume
    if resume_meta["max_steps"] != MAX_TRAIN_SAMPLES // (PER_DEVICE_BS * world * GRAD_ACCUM):
        raise SystemExit(f"checkpoint was made for max_steps={resume_meta['max_steps']} - keep MAX_TRAIN_SAMPLES/BS/GRAD_ACCUM "
                         f"identical across sessions (or set RESUME=False to start over)")
    resume_state = torch.load(resume_dir / "trainer_state.pt", map_location="cpu", weights_only=False)
    log(f"Resuming from {resume_dir} at step {resume_meta['step']}")
    adopt_previous_run(resume_dir)
dist.barrier()

# ----------------------------------------------------------------------------- model
def find_text_model(m):
    for path in ("model.text_model", "text_model", "model.language_model", "language_model"):
        obj = m
        try:
            for a in path.split("."): obj = getattr(obj, a)
        except AttributeError:
            continue
        if hasattr(obj, "layers"): return obj
    raise RuntimeError("cannot locate the LLM decoder (.layers) inside the model")


def truncate_llm(m, n):
    tm = find_text_model(m)
    if n and len(tm.layers) > n:
        tm.layers = torch.nn.ModuleList(list(tm.layers)[:n])
        for cfg in {id(c): c for c in (m.config.text_config, getattr(tm, "config", None)) if c is not None}.values():
            cfg.num_hidden_layers = n
            if getattr(cfg, "layer_types", None): cfg.layer_types = list(cfg.layer_types)[:n]
    return len(tm.layers)


src = str(resume_dir) if resume_dir else MODEL_ID
model = AutoModelForImageTextToText.from_pretrained(src, attn_implementation="sdpa")
n_layers_before = len(find_text_model(model).layers)
n_layers = truncate_llm(model, KEEP_LLM_LAYERS)
model.float(); model.config.use_cache = False
if GRAD_CKPT: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if FREEZE_VISION:
    for n, p in model.named_parameters():
        if "vision_model" in n: p.requires_grad = False
model.to(dev)
n_total = sum(p.numel() for p in model.parameters()); n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"LLM layers {n_layers_before} -> {n_layers} | params: total {n_total/1e6:.1f}M | trainable {n_train/1e6:.1f}M | "
    f"GPU mem after load {torch.cuda.memory_allocated(dev)/2**30:.2f} GB")
model = DDP(model, device_ids=[local_rank], find_unused_parameters=False, broadcast_buffers=False)

# ----------------------------------------------------------------------------- optimizer / schedule
groups = {}
for n, p in model.named_parameters():
    if not p.requires_grad: continue
    vis = "vision_model" in n; nodecay = p.ndim < 2 or n.endswith(".bias")
    groups.setdefault((vis, nodecay), []).append(p)
param_groups = [{"params": ps, "lr": LR * (VISION_LR_MULT if vis else 1.0), "weight_decay": 0.0 if nd else WEIGHT_DECAY}
                for (vis, nd), ps in groups.items()]
if OPTIM == "adamw8bit":
    import bitsandbytes as bnb
    opt = bnb.optim.AdamW8bit(param_groups, betas=(0.9, 0.999))
else:
    opt = torch.optim.AdamW(param_groups, betas=(0.9, 0.999), fused=True)

STEP_SAMPLES = PER_DEVICE_BS * world * GRAD_ACCUM
MAX_STEPS = MAX_TRAIN_SAMPLES // STEP_SAMPLES
sched = get_cosine_schedule_with_warmup(opt, WARMUP_STEPS, MAX_STEPS)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
step, best_acc = 0, -1.0
if resume_state:
    opt.load_state_dict(resume_state["opt"]); sched.load_state_dict(resume_state["sched"])
    scaler.load_state_dict(resume_state["scaler"]); step = resume_state["step"]
    best_acc = resume_state.get("best_acc", -1.0)
    del resume_state; trim_host_memory()
params = [p for p in model.parameters() if p.requires_grad]

# ----------------------------------------------------------------------------- bookkeeping
cfg = {k: (str(v) if isinstance(v, Path) else v) for k, v in globals().items()
       if k.isupper() and isinstance(v, (int, float, str, bool, Path))}
cfg.update(world_size=world, answer_index_base=ANS_BASE, max_steps=MAX_STEPS, effective_batch=STEP_SAMPLES,
           llm_layers=n_layers, torch=torch.__version__, gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if is_main:
    (OUT_DIR / "config.json").write_text(json.dumps(cfg, indent=2))
    try:
        from torch.utils.tensorboard import SummaryWriter
        tb = SummaryWriter(str(OUT_DIR / "tb"))
    except Exception:
        tb = None
    jsonl = open(OUT_DIR / "log_history.jsonl", "a"); csv_path = OUT_DIR / "log_history.csv"
log(f"answer base={ANS_BASE} | max_steps={MAX_STEPS} | effective batch={STEP_SAMPLES} | samples={MAX_STEPS*STEP_SAMPLES} | start step={step}")


def record(d):
    if not is_main: return
    jsonl.write(json.dumps(d) + "\n"); jsonl.flush()
    new = not csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(d.keys()), extrasaction="ignore")
        if new: w.writeheader()
        w.writerow(d)
    if tb:
        for k, v in d.items():
            if k != "step" and isinstance(v, (int, float)): tb.add_scalar(k, v, d["step"])


def save_weights(tag, half):
    """half=True: fp16 copy built tensor-by-tensor on GPU (no big host copy), ~1 GB on disk. Truncated config is saved too."""
    if is_main:
        tmp, final = OUT_DIR / f"{tag}_tmp", OUT_DIR / tag
        shutil.rmtree(tmp, ignore_errors=True)
        net = model.module
        sd = ({k: (v.detach().to(torch.float16) if v.is_floating_point() else v.detach())
               for k, v in net.state_dict().items()} if half else None)
        net.save_pretrained(tmp, state_dict=sd, safe_serialization=True, max_shard_size="2GB" if half else "500MB")
        processor.save_pretrained(tmp)
        del sd
        shutil.rmtree(final, ignore_errors=True); tmp.rename(final)
        log(f"saved {final} ({'fp16' if half else 'fp32'})")
    dist.barrier(); trim_host_memory(); torch.cuda.empty_cache()


def save_resume(feeder):
    tmp, final = OUT_DIR / "last_tmp", OUT_DIR / "last"
    if is_main:
        shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    dist.barrier()
    torch.save(feeder.state, tmp / f"data_state_{rank}.pt")           # every rank stores its own stream position
    dist.barrier()
    if is_main:
        model.module.save_pretrained(tmp, safe_serialization=True, max_shard_size="500MB"); processor.save_pretrained(tmp)
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "step": step, "best_acc": best_acc}, tmp / "trainer_state.pt")
        (tmp / "meta.json").write_text(json.dumps({"step": step, "max_steps": MAX_STEPS, "world": world, "best_acc": best_acc}))
        shutil.rmtree(final, ignore_errors=True); tmp.rename(final)
        log(f"saved {final} (resumable, step {step})")
    dist.barrier(); trim_host_memory(); torch.cuda.empty_cache()


def hub_upload(names):
    if not (HUB_REPO and is_main): return
    try:
        from huggingface_hub import HfApi
        api = HfApi(); api.create_repo(HUB_REPO, private=True, exist_ok=True)
        for nme in names:
            if (OUT_DIR / nme).exists():
                api.upload_folder(folder_path=str(OUT_DIR / nme), path_in_repo=nme, repo_id=HUB_REPO)
                log(f"uploaded {nme}/ to hub {HUB_REPO}")
    except Exception as e:
        log(f"hub upload failed ({type(e).__name__}: {e}) - checkpoint is still in {OUT_DIR}")

# ----------------------------------------------------------------------------- eval
@torch.no_grad()
def run_eval(net, rows):
    """Greedy-decode the answer letter. Rows are split across ranks by index; results are all-reduced."""
    net.eval()
    tot, cor, invalid = [0] * 26, [0] * 26, 0
    def flush(buf):
        nonlocal invalid
        inputs, gold = collate_eval(buf)
        if not gold: return
        inputs = {k: v.to(dev) for k, v in inputs.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out = net.generate(**inputs, max_new_tokens=3, do_sample=False, use_cache=True)
        txt = eval_processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for t, g in zip(txt, gold):
            m = re.search(r"[A-Z]", t.upper()); gi = ord(g) - 65
            tot[gi] += 1
            if m is None: invalid += 1
            elif m.group(0) == g: cor[gi] += 1
    buf = []
    for i, ex in enumerate(rows):
        if i % world != rank: continue
        buf.append(ex)
        if len(buf) == EVAL_BS: flush(buf); buf = []
    if buf: flush(buf)
    s = torch.tensor([invalid] + tot + cor, device=dev, dtype=torch.float64); dist.all_reduce(s)
    s = s.tolist(); inv, tot, cor = s[0], s[1:27], s[27:53]
    n = sum(tot)
    return {"acc": sum(cor) / max(n, 1), "invalid_rate": inv / max(n, 1), "n": int(n),
            "majority_baseline": max(tot) / max(n, 1),
            "acc_by_gold_letter": {chr(65 + i): round(cor[i] / tot[i], 4) for i in range(26) if tot[i]},
            "n_by_gold_letter": {chr(65 + i): int(tot[i]) for i in range(26) if tot[i]}}


log(f"building validation set ({N_VAL} rows from test split)")
val_rows = [compress_row(ex) for ex in split_rows(0, N_VAL)]

# ----------------------------------------------------------------------------- data feeder (resumable)
data_state = None
if resume_dir and (resume_dir / f"data_state_{rank}.pt").exists():
    data_state = torch.load(resume_dir / f"data_state_{rank}.pt", map_location="cpu", weights_only=False)
elif resume_dir:
    log(f"[rank {rank}] no data_state file (world size changed?) -> epoch restarts with a new shuffle; some rows will repeat")
start_epoch = (step * STEP_SAMPLES) // TRAIN_ROWS + (1000 if (resume_dir and data_state is None) else 0)   # distinct seed if position is lost
feeder = Feeder(data_state, start_epoch, make_stream, collate_train, SEED, SHUFFLE_BUFFER, PER_DEVICE_BS, PREFETCH, log_fn=log)

# ----------------------------------------------------------------------------- train
import psutil
proc = psutil.Process()
t0 = time.time(); t_win = time.time()
win_loss = win_n = 0.0; win_samples = 0
win_corr = torch.zeros((), device=dev); win_cnt = torch.zeros((), device=dev)
model.train(); opt.zero_grad(set_to_none=True)
timed_out = False
log("starting training")

while step < MAX_STEPS:
    step_loss = 0.0
    for m in range(GRAD_ACCUM):
        batch = feeder.next()
        if step == 0 and m == 0 and is_main:
            s = re.sub(r"(<image>)+", "<image...>", tok.decode(batch["input_ids"][0]))
            sup = tok.decode(batch["input_ids"][0][batch["labels"][0] != -100])
            log(f"SANITY first sample:\n{s}\n--- supervised tokens -> {sup!r}  (expect: a single letter + <end_of_utterance>)")
        batch = {k: v.to(dev, non_blocking=True) for k, v in batch.items()}
        ctx = model.no_sync() if m < GRAD_ACCUM - 1 else contextlib.nullcontext()
        with ctx:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**batch)
                loss = out.loss / GRAD_ACCUM
            scaler.scale(loss).backward()
        with torch.no_grad():                                   # running train accuracy on the answer letter
            lb = batch["labels"][:, 1:]; sel = (lb != -100) & (lb != EOU_ID)
            if sel.any():
                win_corr += (out.logits[:, :-1][sel].argmax(-1) == lb[sel]).sum(); win_cnt += sel.sum()
        step_loss += loss.detach().float().item()
        del batch, loss, out
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM).item()
    scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    step += 1
    win_loss += step_loss; win_n += 1; win_samples += STEP_SAMPLES

    if step % LOG_EVERY == 0 or step == MAX_STEPS:
        avg = all_mean(win_loss / win_n); now = time.time()
        sc = torch.stack([win_corr, win_cnt]); dist.all_reduce(sc); tr_acc = (sc[0] / sc[1].clamp(min=1)).item()
        trim_host_memory()
        rec = {"step": step, "samples_seen": step * STEP_SAMPLES, "epoch": round(step * STEP_SAMPLES / TRAIN_ROWS, 4),
               "loss": round(avg, 5), "train_acc": round(tr_acc, 4), "lr": sched.get_last_lr()[0], "grad_norm": round(gnorm, 4),
               "loss_scale": scaler.get_scale() if USE_AMP else 1.0,
               "samples_per_sec": round(win_samples / (now - t_win), 2),
               "gpu_peak_gb": round(torch.cuda.max_memory_allocated(dev) / 2**30, 2),
               "host_rss_gb": round(all_mean(proc.memory_info().rss / 2**30), 2),
               "host_free_gb": round(psutil.virtual_memory().available / 2**30, 1),
               "elapsed_min": round((now - t0) / 60, 1)}
        log(" | ".join(f"{k}={v}" for k, v in rec.items()))
        record(rec); win_loss = win_n = 0.0; win_samples = 0; win_corr.zero_(); win_cnt.zero_(); t_win = now

    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        r = run_eval(model.module, val_rows); model.train()
        improved = r["acc"] > best_acc
        log(f"VALID step {step}: acc={r['acc']:.4f} (majority baseline {r['majority_baseline']:.3f}) "
            f"invalid={r['invalid_rate']:.3f} n={r['n']} {'<- new best' if improved else ''}")
        record({"step": step, "val_acc": round(r["acc"], 5), "val_invalid_rate": round(r["invalid_rate"], 5), "val_n": r["n"]})
        if improved:
            best_acc = r["acc"]; save_weights("best", half=True)

    stop = torch.tensor([float(is_main and (time.time() - T_START) > MAX_HOURS * 3600)], device=dev)
    dist.all_reduce(stop, op=dist.ReduceOp.MAX)
    if step % SAVE_EVERY == 0 or stop.item() > 0:
        save_resume(feeder)
    if stop.item() > 0 and step < MAX_STEPS:
        timed_out = True
        log("time budget reached -> stopping. Save Version, attach this notebook's output as an input, run the same cell again.")
        break

feeder.close()
if timed_out:
    hub_upload(["last", "best"])
    if is_main: jsonl.close()
    dist.barrier(); dist.destroy_process_group()
    raise SystemExit(0)                                   # no final test: not finished yet, and the clock is running

# ----------------------------------------------------------------------------- finish: final test with BEST model
save_weights("final", half=True)
if is_main:
    jsonl.flush()
del opt, scaler, sched, params, groups, param_groups, model, feeder
trim_host_memory(); torch.cuda.empty_cache()

if (time.time() - T_START) > (MAX_HOURS + 0.6) * 3600:        # finished, but too close to Kaggle's 12 h wall to start the test
    log("training finished but the clock is nearly out -> skipping the final TEST (run a separate eval on best/)")
    hub_upload(["best", "final"])
    if is_main: jsonl.close()
    dist.barrier(); dist.destroy_process_group(); raise SystemExit(0)

best_dir = OUT_DIR / ("best" if (OUT_DIR / "best").exists() else "final")
log(f"final TEST with weights from {best_dir.name}/")
test_net = AutoModelForImageTextToText.from_pretrained(str(best_dir), attn_implementation="sdpa").float().to(dev).eval()
n_exp = len(find_text_model(test_net).layers)
log(f"export check: {best_dir.name}/ loads with {n_exp} LLM layers, config.text_config.num_hidden_layers="
    f"{test_net.config.text_config.num_hidden_layers}")
test_res = run_eval(test_net, split_rows(N_VAL, (N_VAL + N_TEST) if N_TEST else 0))
test_res.update(weights=best_dir.name, best_val_acc=best_acc, llm_layers=n_exp)
log(f"TEST: acc={test_res['acc']:.4f} (majority baseline {test_res['majority_baseline']:.3f}) "
    f"invalid={test_res['invalid_rate']:.3f} n={test_res['n']}")
if is_main:
    (OUT_DIR / "test_results.json").write_text(json.dumps(test_res, indent=2))
    record({"step": step, "test_acc": round(test_res["acc"], 5), "test_n": test_res["n"]})
    (OUT_DIR / "SMOLVLA_USAGE.md").write_text(f"""# Using this backbone in SmolVLA (LeRobot)
Weights: best/ (best validation accuracy) and final/ (last step, fully annealed). Both fp16, {n_exp} LLM layers.

    lerobot-train --policy.type=smolvla \\
      --policy.vlm_model_name=<hub repo id or path to best/> \\
      --policy.load_vlm_weights=true \\
      --policy.num_vlm_layers={n_exp} \\
      --policy.freeze_vision_encoder=true --policy.train_expert_only=true \\
      --dataset.repo_id=<your dataset> ...

Do NOT start from --policy.path=lerobot/smolvla_base: its action expert was trained against the ORIGINAL SmolVLM2
features, so with this backbone the expert has to be trained again. Flag names can differ between LeRobot versions - check
`lerobot-train --help`.  Images: letterboxed to {IMG_SIZE}x{IMG_SIZE} (RESIZE_MODE={RESIZE_MODE}), no tiling.
""")
    hist = [json.loads(l) for l in open(OUT_DIR / "log_history.jsonl")]
    (OUT_DIR / "history.json").write_text(json.dumps({"config": cfg, "log_history": hist, "test": test_res}, indent=2))
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        tr = [h for h in hist if "loss" in h]; ev = [h for h in hist if "val_acc" in h]
        fig, ax = plt.subplots(1, 3, figsize=(16, 4))
        ax[0].plot([h["step"] for h in tr], [h["loss"] for h in tr]); ax[0].set_title("train loss (answer letter)")
        ax[1].plot([h["step"] for h in tr], [h["train_acc"] for h in tr], label="train (running)", alpha=.6)
        ax[1].plot([h["step"] for h in ev], [h["val_acc"] for h in ev], marker="o", label="validation"); ax[1].legend(); ax[1].set_title("accuracy")
        ax[2].plot([h["step"] for h in tr], [h["lr"] for h in tr]); ax[2].set_title("lr")
        for a in ax: a.set_xlabel("step"); a.grid(alpha=.3)
        plt.tight_layout(); plt.savefig(OUT_DIR / "curves.png", dpi=120)
    except Exception as e:
        log(f"plot failed: {e}")
    if tb: tb.close()
    jsonl.close()
    log(f"DONE. best val acc={best_acc:.4f} | test acc={test_res['acc']:.4f} | outputs in {OUT_DIR}")
hub_upload(["best", "final"])
dist.barrier(); dist.destroy_process_group()


In [ ]:
import os, shutil
FRESH_START = False     # True ONLY for a brand-new run: wipes this session's output folder (never touches /kaggle/input)
OUT = '/kaggle/working/robo2vlm_smolvlm2_run'
if FRESH_START: shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)
!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 train_v3.py 2>&1 | tee -a /kaggle/working/robo2vlm_smolvlm2_run/stdout.log


In [ ]:
!ls -lh /kaggle/working/robo2vlm_smolvlm2_run /kaggle/working/robo2vlm_smolvlm2_run/best
!tail -n 5 /kaggle/working/robo2vlm_smolvlm2_run/train.log
!cat /kaggle/working/robo2vlm_smolvlm2_run/test_results.json   # only exists once the whole epoch is finished
from IPython.display import Image, display
import os
p='/kaggle/working/robo2vlm_smolvlm2_run/curves.png'
if os.path.exists(p): display(Image(p))
